# Ranker Dataset and Baseline Training

Membangun synthetic ranking data dari hasil retrieval embedding di atas dataset raw yang sudah diprepare, lalu melatih baseline `LGBMRanker`.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMRanker, early_stopping, log_evaluation
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from src.config import ARTIFACTS_DIR, MODELS_DIR, INTERIM_DIR
from src.evaluation import evaluate_grouped_ndcg
from src.features import build_candidate_features, compute_relevance, feature_columns, synthetic_user_profiles
from src.preprocessing import load_prepared_dataset, save_dataframe


c:\porto\NemuParfang\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def expand_profiles(base_profiles):
    variants = [
        ('core', ''),
        ('signature', ' versatile signature scent'),
        ('casual', ' easy daily wear'),
        ('intense', ' stronger longer lasting version'),
    ]
    expanded = []
    for profile in base_profiles:
        family = profile.get('profile_family', profile['profile_id'])
        for suffix, extra_text in variants:
            expanded.append({
                **profile,
                'profile_id': f"{family}_{suffix}",
                'profile_family': family,
                'profile_text': profile['profile_text'] + extra_text,
            })
    return expanded

df = load_prepared_dataset(INTERIM_DIR)
perfume_embeddings = joblib.load(ARTIFACTS_DIR / 'perfume_embeddings.joblib')
with (ARTIFACTS_DIR / 'embedding_config.json').open('r', encoding='utf-8') as handle:
    embedding_config = json.load(handle)
encoder = SentenceTransformer(embedding_config['model_name'])
print('Loaded perfumes:', len(df))
print('Embedding shape:', perfume_embeddings.shape)
print('Embedding model:', embedding_config['model_name'])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7825.19it/s]


Loaded perfumes: 70103
Embedding shape: (70103, 384)
Embedding model: all-MiniLM-L6-v2


In [3]:
rng = np.random.default_rng(42)
all_indices = np.arange(len(df))
profile_frames = []
for profile in expand_profiles(synthetic_user_profiles()):
    user_vec = encoder.encode(profile['profile_text'])
    scores = cosine_similarity([user_vec], perfume_embeddings).ravel()
    ranked_indices = np.argsort(scores)[::-1]
    top_indices = ranked_indices[:80]
    hard_negative_pool = ranked_indices[80:280]
    hard_negative_indices = rng.choice(hard_negative_pool, size=min(80, len(hard_negative_pool)), replace=False)
    random_negative_indices = rng.choice(all_indices, size=80, replace=False)
    candidate_indices = np.unique(np.concatenate([top_indices, hard_negative_indices, random_negative_indices]))

    candidates = df.iloc[candidate_indices].copy().reset_index(drop=True)
    candidates['profile_id'] = profile['profile_id']
    candidates['profile_family'] = profile['profile_family']
    candidates['cosine_similarity_score'] = scores[candidate_indices]
    candidates = build_candidate_features(candidates, profile)
    candidates['relevance'] = candidates.apply(lambda row: compute_relevance(profile, row), axis=1)
    profile_frames.append(candidates)

rank_df = pd.concat(profile_frames, ignore_index=True)
display(rank_df[['profile_id', 'profile_family', 'perfume_name', 'brand', 'country', 'cosine_similarity_score', 'relevance']].head(15))
print('Rank frame rows:', len(rank_df))
print('Profiles:', rank_df['profile_id'].nunique())
print('Profile families:', rank_df['profile_family'].nunique())
display(rank_df['relevance'].value_counts().sort_index().to_frame('count'))


,profile_id,profile_family,perfume_name,brand,country,cosine_similarity_score,relevance
0,woody_evening_core,woody_evening,Aures,Avon,United States,0.694916,2
1,woody_evening_core,woody_evening,Hombre,Avon,United States,0.666430,1
2,woody_evening_core,woody_evening,Sassy Swirls Vanilla Bean,Avon,United States,0.688209,2
3,woody_evening_core,woody_evening,Black Essential Real,Avon,United States,0.476207,0
4,woody_evening_core,woody_evening,Full Speed Nitro,Avon,United States,0.695302,1
5,woody_evening_core,woody_evening,Herve Leger Ete,Avon,United States,0.408458,1
6,woody_evening_core,woody_evening,Mesmerize Mystique Amber,Avon,United States,0.666995,1
7,woody_evening_core,woody_evening,Azzaro Pour Homme Limited Edition 2014,Azzaro,NaN,0.692069,2
8,woody_evening_core,woody_evening,Wicked Vanilla Woods,Bath & Body Works,United States,0.670078,2
9,woody_evening_core,woody_evening,Amber Oud,By Kilian,NaN,0.689570,3


Rank frame rows: 14385
Profiles: 60
Profile families: 15


,count
relevance,
0,3119
1,2360
2,5293
3,3105
4,508


In [4]:
saved_path = save_dataframe(rank_df, ARTIFACTS_DIR / 'ranker_training_frame.csv')
print('Saved training frame to:', saved_path)


Saved training frame to: C:\porto\NemuParfang\ml_training_notebooks\artifacts\ranker_training_frame.csv


In [5]:
families = rank_df['profile_family'].unique().tolist()
train_families, val_families = train_test_split(families, test_size=0.33, random_state=42)
train_df = rank_df[rank_df['profile_family'].isin(train_families)].copy()
val_df = rank_df[rank_df['profile_family'].isin(val_families)].copy()
train_groups = train_df.groupby('profile_id').size().tolist()
val_groups = val_df.groupby('profile_id').size().tolist()
print('Train rows:', len(train_df), 'Validation rows:', len(val_df))
print('Train groups:', len(train_groups), 'Validation groups:', len(val_groups))

ranker = LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=800,
    num_leaves=31,
    learning_rate=0.03,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.05,
    reg_lambda=0.1,
    force_col_wise=True,
    random_state=42,
)

ranker.fit(
    train_df[feature_columns()],
    train_df['relevance'],
    group=train_groups,
    eval_set=[(val_df[feature_columns()], val_df['relevance'])],
    eval_group=[val_groups],
    eval_at=[5, 10],
    callbacks=[early_stopping(50), log_evaluation(50)],
)

metric = evaluate_grouped_ndcg(ranker, val_df[feature_columns()], val_df['relevance'].to_numpy(), val_groups, k=10)
print('Validation NDCG@10:', metric)


Train rows: 9589 Validation rows: 4796
Train groups: 40 Validation groups: 20
[LightGBM] [Info] Total Bins 1288
[LightGBM] [Info] Number of data points in the train set: 9589, number of used features: 13
Training until validation scores don't improve for 50 rounds
[50]	valid_0's ndcg@5: 0.996501	valid_0's ndcg@10: 0.982663
Early stopping, best iteration is:
[15]	valid_0's ndcg@5: 0.996501	valid_0's ndcg@10: 0.97239
Validation NDCG@10: 0.9862904308103593


In [6]:
importance_df = pd.DataFrame({
    'feature': feature_columns(),
    'importance': ranker.feature_importances_,
}).sort_values('importance', ascending=False)
display(importance_df)
joblib.dump(ranker, MODELS_DIR / 'lgbm_ranker_baseline.joblib')
print('Baseline ranker saved to outputs/models/.')


,feature,importance
12,freshness_score,74
8,rating_norm,66
0,cosine_similarity_score,61
3,note_overlap_ratio,40
7,year_window_match,40
11,popularity_score,37
1,accord_overlap_ratio,34
10,year_norm,33
4,gender_match,30
9,review_count_norm,17


Baseline ranker saved to outputs/models/.
